# NFT（Negative-aware Fine-Tuning）代码实现详解

> **论文**: *Bridging Supervised Learning and Reinforcement Learning in Math Reasoning*  
> **arXiv**: 2505.18116 (NVIDIA / Tsinghua, 2025)

## 核心思想

NFT 通过**隐式负策略（Implicit Negative Policy）**，用纯监督学习的方式同时利用正样本和负样本来优化 LLM，在数学上证明了与 GRPO 等强化学习算法在同策略下的等价性。

**本 Demo 覆盖的核心公式**：
1. **全概率公式策略拆分**：$\pi_{\text{old}} = c \cdot \pi^+ + (1-c) \cdot \pi^-$
2. **隐式负策略**：$\text{proxy} = (1 - c \cdot R_\theta) / (1 - c)$
3. **NFT 损失函数**：$\mathcal{L} = r \cdot [-\log R_\theta] + (1-r) \cdot [-\log \text{proxy}]$
4. **STE 截断防爆**：前向拦截 + 后向直通
5. **难度感知归一化**：$\omega(q)$ 加权

下面逐公式拆解代码实现 👇

## 导入依赖与超参数设置

导入 PyTorch 相关库并设置所有超参数，对应论文代码中的 `config/nft_trainer.yaml` 配置文件。

关键超参数说明：
- `NEG_WEIGHT`：控制算法行为的核心开关 — `1.0` 为 NFT（利用正负样本），`0.0` 为 RFT（只用正样本）
- `NORMALIZE`：难度感知归一化方式 — 对应论文中 $\omega(q)$ 的不同选择
- `CLAMP_POSITIVE / CLAMP_NEGATIVE`：STE 截断的阈值，防止数值爆炸

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)   # 固定随机种子, 保证结果可复现

# ---- 超参数 (对应 config/nft_trainer.yaml) ----
BATCH_SIZE     = 8       # 一次采样的 response 总数
N_PROMPTS      = 2       # prompt 数 (每个 prompt 采 4 个 response)
N_PER_PROMPT   = BATCH_SIZE // N_PROMPTS   # 每个 prompt 的 response 数
SEQ_LEN        = 16      # response 序列长度 (token 数)
VOCAB_SIZE     = 100     # 词汇表大小
NEG_WEIGHT     = 1.0     # 负样本权重: 1.0=NFT, 0.0=RFT, -1.0=DAPO
NORMALIZE      = 1       # 归一化方式: 0=无, 1=Dr.GRPO, 2=标准GRPO
CLAMP_POSITIVE = 0.0     # 正样本 ratio 的下界 (clamp 截断阈值)
CLAMP_NEGATIVE = 1.0     # 负样本 proxy 的下界
ENTROPY_COEFF  = 0.01    # 熵正则系数 β_ent (鼓励探索)
KL_COEFF       = 0.0     # KL 散度系数 β_kl (本 demo 不使用 ref policy)
LEARNING_RATE  = 1e-3    # Adam 学习率
EPSILON        = 1e-6    # 数值稳定性的小量 (防止除零)

## STE 截断函数（Straight-Through Estimator）

这是 NFT 代码中最关键的工程技巧之一，对应论文公式 10 中的 $\text{max\_v}(\cdot, \epsilon)$ 算子。

**问题**：NFT 的隐式负策略 proxy $= (1 - c \cdot R_\theta) / (1 - c)$ 在训练中可能出现极端值，导致 $\log(\le 0) \to \text{NaN}$，直接让训练崩溃。

**方案**：用 `torch.clamp` 截断危险值，但普通 clamp 在截断时梯度为 0（"梯度断流"）。通过 **STE（直通估计器）** 实现"前向拦截，后向直通"：
- **前向传播**：正常使用截断后的值，防止 NaN
- **反向传播**：假装截断没发生，梯度原封不动传回去

$$\text{clamp\_preserve\_grad}(x) = x + (\text{clamp}(x) - x).\text{detach}()$$

In [ ]:
def clamp_preserve_grad(x, min_val=None, max_val=None):
    """Straight-Through Estimator (STE) 版本的 clamp
    
    核心思想: 前向传播用 clamp 后的值 (防止数值爆炸),
             反向传播梯度直接流过原始 x (防止梯度断流)。
    
    实现原理:
        clipped = clamp(x)                    # 截断后的值
        return x + (clipped - x).detach()     # 前向 = clipped, 反向 = x 的梯度
    
    为什么需要 STE?
    - 普通 torch.clamp 在触发截断时梯度为 0, 会导致"梯度断流"
    - NFT 的 proxy 项在训练中可能出现极端值, 需要截断来防爆
    - 但截断后又不能丢失梯度, 否则模型无法从错误样本中学习
    - STE 完美解决了这个矛盾: "前向拦截, 后向直通"
    """
    clipped = torch.clamp(x, min=min_val, max=max_val)
    return x + (clipped - x).detach()   # 前向是 clipped 值, 反向梯度流经原始 x

## 构造玩具模型（ToyPolicy）

定义一个极简的策略网络来模拟真实的 actor policy。结构为 `Embedding → Linear → Logits`，与真实 LLM 的 Transformer 结构类似但大幅简化。

该模型提供三个核心方法：
- `forward()`：前向传播，输出 logits
- `get_log_probs()`：计算每个 token 的 $\log \pi_\theta(a|s)$，用于策略比率计算
- `get_entropy()`：计算策略熵 $H(\pi)$，用于熵正则化

In [ ]:
class ToyPolicy(nn.Module):
    """极简策略网络: embedding → linear → logits
    
    模拟一个 actor policy, 结构与真实 LLM 类似但大幅简化:
    - Embedding 层: 将 token id 映射到隐藏向量
    - Linear 层: 将隐藏向量映射回词汇表大小的 logits
    """
    def __init__(self, vocab_size, hidden=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)   # 词嵌入层
        self.head  = nn.Linear(hidden, vocab_size)      # 输出头: 生成每个 token 的 logits

    def forward(self, input_ids):
        """前向传播: 返回每个位置的 logits
        输入: input_ids (bs, seq) — token id 序列
        输出: logits (bs, seq, vocab) — 每个位置对词汇表中每个 token 的未归一化分数
        """
        h = self.embed(input_ids)       # (bs, seq) → (bs, seq, hidden)
        logits = self.head(h)           # (bs, seq, hidden) → (bs, seq, vocab)
        return logits

    def get_log_probs(self, input_ids):
        """计算每个 token 的对数概率 log π(a|s)
        
        步骤:
        1. forward 得到 logits
        2. log_softmax 归一化为对数概率分布
        3. gather 取出实际生成的 token 对应的 log prob
        """
        logits = self.forward(input_ids)                    # (bs, seq, vocab)
        log_probs = F.log_softmax(logits, dim=-1)           # 对数概率分布
        # 用 gather 取出 input_ids 对应位置的 log prob
        token_log_probs = log_probs.gather(
            dim=-1, index=input_ids.unsqueeze(-1)
        ).squeeze(-1)                                        # (bs, seq)
        return token_log_probs

    def get_entropy(self, input_ids):
        """计算策略熵 H(π) = -Σ p(a) log p(a)
        
        策略熵衡量模型输出的"不确定性":
        - 熵大 → 策略更随机, 探索性强
        - 熵小 → 策略更确定, 利用性强
        """
        logits = self.forward(input_ids)
        probs = F.softmax(logits, dim=-1)                   # 概率分布
        log_probs = F.log_softmax(logits, dim=-1)           # 对数概率分布
        # H(π) = -Σ p(a) · log p(a)
        entropy = -(probs * log_probs).sum(dim=-1)          # (bs, seq)
        return entropy

## 模拟数据生成（Rollout 阶段）

在真实的 NFT 训练中，模型需要对每个 prompt 进行 **rollout**（采样多个 response），然后用外部验证器（如 Python 代码解释器）打分。

本 demo 中我们用随机数据模拟这个过程：
- **2 个 prompt**，每个 prompt 采样 **4 个 response**
- Prompt 0 是"简单题"（75% 正确率），Prompt 1 是"难题"（25% 正确率）
- `actor` 是当前正在训练的策略 $\pi_\theta$，`actor_old` 是采样时的旧策略 $\pi_{\theta_{\text{old}}}$（参数冻结）

In [ ]:
print("=" * 70)
print("NFT Minimal Demo — 模拟一个完整的训练 step")
print("=" * 70)

# 创建两个策略网络
actor     = ToyPolicy(VOCAB_SIZE)     # π_θ: 正在训练的策略 (参数会被更新)
actor_old = ToyPolicy(VOCAB_SIZE)     # π_θ_old: 采样时的旧策略 (参数冻结, 不更新)
actor_old.load_state_dict(actor.state_dict())   # 初始时两者参数完全相同

# Adam 优化器
optimizer = torch.optim.Adam(actor.parameters(), lr=LEARNING_RATE)

# ---- 模拟 rollout 阶段 ----
# 每个 prompt 生成 N_PER_PROMPT 个 response (随机生成 token 序列)
responses     = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
# prompt_id 标记每个 response 属于哪个 prompt: [0,0,0,0, 1,1,1,1]
prompt_id     = torch.arange(N_PROMPTS).repeat_interleave(N_PER_PROMPT)
# response_mask: 标记哪些位置有效 (简化起见, 全部为 1)
response_mask = torch.ones(BATCH_SIZE, SEQ_LEN)

# 模拟 verifier 打分: reward ∈ {+1 (正确), -1 (错误)}
# prompt 0: 3对1错 (correct_rate=0.75) — 相对简单的题
# prompt 1: 1对3错 (correct_rate=0.25) — 相对困难的题
rewards = torch.tensor([
    +1.0, +1.0, +1.0, -1.0,   # prompt 0: 正确率 75%
    +1.0, -1.0, -1.0, -1.0,   # prompt 1: 正确率 25%
])

print(f"\n[数据] 每个 prompt 生成 {N_PER_PROMPT} 个 response")
print(f"[数据] Rewards: {rewards.tolist()}")
print(f"[数据] Prompt 0 正确率: 75%, Prompt 1 正确率: 25%")

### 公式 1：二值正确率掩码（correct_mask）

将 reward 从 $\{-1, +1\}$ 映射为 $\{0, 1\}$ 的二值标签：

$$\text{correct\_mask}_i = \frac{r_i + 1}{2}$$

这个掩码在后续作为"开关"，区分正样本和负样本的损失项。

In [ ]:
# 将 reward ∈ {-1, +1} 映射为二值标签 ∈ {0, 1}
# +1 (正确) → 1.0,  -1 (错误) → 0.0
correct_mask = (rewards + 1.0) / 2.0           # shape: (BATCH_SIZE,)

# 增加一个维度, 便于后续与 (bs, seq) 形状的张量做广播运算
correct_mask = correct_mask.unsqueeze(-1)       # shape: (BATCH_SIZE, 1)

print(f"[公式1] correct_mask = (reward+1)/2: {correct_mask.squeeze().tolist()}")

### 公式 2：GRPO 优势估计与正确率计算

对同一 prompt 下的所有 response 做**组内归一化**：

$$A_i = \frac{r_i - \text{mean}(\text{group})}{\text{std}(\text{group}) + \epsilon}$$

$$c = \frac{\text{mean}(\text{group}) + 1}{2}$$

- $A_i$ 是 **GRPO advantage**：衡量某个 response 在同组内的相对好坏
- $c$ 是 **correct_rate**：该 prompt 的经验正确率，后续用于隐式负策略和难度加权

**关键点**：advantage 是组内相对比较，不是绝对值。一个 response 好不好，取决于它比同组其他 response 好多少。

In [ ]:
# 初始化优势和正确率张量
advantages   = torch.zeros(BATCH_SIZE)
correct_rate = torch.zeros(BATCH_SIZE)

# 对每个 prompt 组分别计算
for pid in range(N_PROMPTS):
    group_mask = (prompt_id == pid)                  # 选出属于同一 prompt 的 response
    group_rewards = rewards[group_mask]

    # 组内均值和标准差
    group_mean = group_rewards.mean()
    group_std  = group_rewards.std() if len(group_rewards) > 1 else torch.tensor(1.0)

    # GRPO advantage: 组内标准化 (减均值除标准差)
    advantages[group_mask] = (group_rewards - group_mean) / (group_std + EPSILON)

    # correct_rate: reward ∈ {-1,+1}, mean(rewards) = 2c - 1, 所以 c = (mean+1)/2
    c = (group_mean + 1.0) / 2.0
    correct_rate[group_mask] = c

# 扩展到序列维度, 同一个 response 的所有 token 共享相同的 advantage 和 correct_rate
advantages   = advantages.unsqueeze(-1).expand_as(response_mask)    # (bs, seq)
correct_rate = correct_rate.unsqueeze(-1).expand_as(response_mask)  # (bs, seq)

print(f"[公式2] GRPO Advantage (prompt 0): {advantages[0, 0].item():.4f}")
print(f"[公式2] correct_rate: prompt 0 = {correct_rate[0, 0].item():.2f}, "
      f"prompt 1 = {correct_rate[4, 0].item():.2f}")

### 公式 3：策略比率（Importance Sampling Ratio）

$$R_\theta(a|s) = \frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} = \exp\big(\log \pi_\theta(a|s) - \log \pi_{\theta_{\text{old}}}(a|s)\big)$$

这是 PPO/GRPO 中的标准重要性采样比率，衡量当前策略相对于采样时旧策略的"偏离程度"：
- $R_\theta = 1$：当前策略与旧策略一致（on-policy）
- $R_\theta > 1$：当前策略更倾向于生成该回答
- $R_\theta < 1$：当前策略更不倾向于生成该回答

在 token 级别计算，即每个 token 都有独立的 ratio 值。

In [ ]:
# 旧策略的 log_probs: 不追踪梯度 (作为 baseline)
with torch.no_grad():
    old_log_probs = actor_old.get_log_probs(responses)    # (bs, seq)

# 当前策略的 log_probs: 追踪梯度 (用于反向传播)
log_probs = actor.get_log_probs(responses)                # (bs, seq)

# 计算 log(ratio) = log(π_θ) - log(π_θ_old)
negative_approx_kl = log_probs - old_log_probs

# 通过 exp 得到 token-level 的 importance sampling ratio
# clamp(max=10.0) 防止 exp 溢出
ratio = torch.exp(negative_approx_kl.clamp(max=10.0))     # (bs, seq)

print(f"[公式3] ratio = exp(logπ - logπ_old)")
print(f"[公式3] ratio 范围: [{ratio.min().item():.4f}, {ratio.max().item():.4f}]")
print(f"        (初始时 actor == actor_old, ratio 应接近 1.0)")

### 公式 4 ⭐：隐式负策略（Implicit Negative Policy）— NFT 的核心创新

这是 NFT 最关键的公式！通过全概率公式的减法拆分，用**正在训练的正策略参数**来构造一个"隐式负策略"：

$$\text{proxy} = \frac{1 - c \cdot R_\theta}{1 - c}$$

其中 $R_\theta = \pi_\theta / \pi_{\theta_{\text{old}}}$ 是 token 级似然比，$c$ 是题目正确率。

**直觉理解**：
- 当 $c \to 0$（难题）：$\text{proxy} \approx R_\theta$，负样本几乎被当作正样本来优化
- 当 $c \to 1$（简单题）：$\text{proxy}$ 对负样本施加强烈惩罚
- 当 $R_\theta$ 增大 → $\text{proxy}$ 减小 → $-\log(\text{proxy})$ 增大 → 惩罚负样本的梯度反向推动 $R_\theta$ 减小

**论文公式 (7) 的推导**：$\pi_{\text{old}} = c \cdot \pi^+ + (1-c) \cdot \pi^-$，解出 $\pi^- = (\pi_{\text{old}} - c \cdot \pi^+) / (1-c)$

In [ ]:
# 对 correct_rate 做 clamp, 防止 c=0 或 c=1 时除零
c = torch.clamp(correct_rate, min=0.02, max=0.98)
negative_rate = 1.0 - c   # (1-c), 后续用于难度感知归一化

# ⭐ 核心公式: 用正策略的 ratio 构造隐式负策略的 proxy
# proxy = (1 - c · ratio) / (1 - c)
# 来源于全概率公式的减法拆分: π⁻ = (π_old - c · π⁺) / (1 - c)
# 除以 π_old 后变为 ratio 形式: proxy = (1 - c · R_θ) / (1 - c)
proxy = (1.0 - c * ratio) / (1.0 - c)

print(f"[公式4] ⭐ 隐式负策略 proxy = (1 - c·ratio) / (1 - c)")
print(f"[公式4] proxy 范围: [{proxy.min().item():.4f}, {proxy.max().item():.4f}]")

### 公式 5：正损失与负损失

分别对正样本和负样本计算对数似然比损失：

$$\mathcal{L}_{\text{pos}} = -\log(\text{clamp}(R_\theta, \min=c_{\text{pos}}))$$

$$\mathcal{L}_{\text{neg}} = -\log(\text{clamp}(\text{proxy}, \min=|c_{\text{neg}}|))$$

- **正样本**：$-\log(\text{ratio})$，当 ratio 增大时损失减小，鼓励模型提高正确回答的概率
- **负样本**：$-\log(\text{proxy})$，当 ratio 增大时 proxy 减小，$-\log(\text{proxy})$ 增大，反向传播会抑制错误回答的概率
- 两者都使用 **STE clamp** 防止数值爆炸同时保持梯度流通

In [ ]:
# 正样本损失: -log(ratio), 鼓励模型增大正样本的生成概率
# 使用 clamp_preserve_grad 确保 ratio 被截断时梯度仍然流通
positive_loss = -torch.log(clamp_preserve_grad(ratio, min_val=CLAMP_POSITIVE))

# 负样本损失: -log(proxy), 通过隐式负策略间接惩罚错误路径
# proxy 减小时, -log(proxy) 增大, 从而加大对错误路径的惩罚
negative_loss = -torch.log(clamp_preserve_grad(proxy, min_val=abs(CLAMP_NEGATIVE)))

print(f"[公式5] L_pos = -log(clamp(ratio)), L_neg = -log(clamp(proxy))")
print(f"        positive_loss 均值: {positive_loss.mean().item():.4f}")
print(f"        negative_loss 均值: {negative_loss.mean().item():.4f}")

### 公式 6 ⭐：NFT 组合损失

将正样本损失和负样本损失通过 `correct_mask` 开关组合成统一的 NFT 损失：

$$\mathcal{L}_{\text{NFT}} = \mathcal{L}_{\text{neg}} \cdot (1 - m) \cdot w + \mathcal{L}_{\text{pos}} \cdot m$$

其中 $m$ 为 `correct_mask`（1=正确，0=错误），$w$ 为 `NEG_WEIGHT`：

| `neg_weight` | 算法 | 行为 |
|---|---|---|
| `1.0` | **NFT** | 正负样本同等权重，同时利用正负样本 |
| `0.0` | **RFT** | 丢弃所有负样本，只用正样本训练 |
| `-1.0` | **DAPO** | 走标准 RL 分支 |

In [ ]:
# 用 correct_mask 作为开关, 将正负损失组合成统一的损失
# correct_mask=1 (正样本) → 只保留 positive_loss
# correct_mask=0 (负样本) → 只保留 negative_loss × NEG_WEIGHT
pg_losses = (negative_loss * (1.0 - correct_mask) * NEG_WEIGHT
           + positive_loss * correct_mask)

print(f"[公式6] ⭐ L_NFT = L_neg·(1-m)·w + L_pos·m")
print(f"        neg_weight={NEG_WEIGHT} (NFT 模式)")
print(f"        当 neg_weight=1.0 → NFT (正负同等权重)")
print(f"        当 neg_weight=0.0 → RFT (丢弃所有负样本)")

### 公式 7：难度感知归一化（Difficulty-aware Normalization）

不同难度的题目应该获得不同的训练权重。论文中对应 $\omega(q)$ 的设计：

| 归一化方式 | 权重公式 | 效果 |
|---|---|---|
| `normalize=0` | 无 | 所有题目等权 |
| `normalize=1` (Dr. GRPO) | $\times (1-c)$ | 难题权重更大，简单题权重更小 |
| `normalize=2` (标准 GRPO) | $\times \sqrt{(1-c)/c}$ | 精确匹配 GRPO 的 on-policy 梯度 |

**核心思想**：正确率 $c$ 低的难题包含更多信息量，应该被赋予更高的权重。这使得 NFT 在宏观层面与 GRPO 的行为对齐。

In [ ]:
# 根据归一化方式对损失进行难度加权
if NORMALIZE == 1:
    # Dr. GRPO 模式: 乘以 (1-c), 难题 (c 小) 权重更大
    pg_losses = pg_losses * negative_rate
    norm_desc = "(1-c) [Dr. GRPO]"
elif NORMALIZE == 2:
    # 标准 GRPO 模式: 乘以 √((1-c)/c), 匹配 GRPO 的 on-policy 行为
    pg_losses = pg_losses * (negative_rate / c) ** 0.5
    norm_desc = "√((1-c)/c) [标准 GRPO]"
else:
    # 不做归一化
    norm_desc = "无归一化"

print(f"[公式7] normalize={NORMALIZE}: L *= {norm_desc}")
print(f"        prompt 0 (c=0.75) 权重: {(1 - 0.75):.2f}")
print(f"        prompt 1 (c=0.25) 权重: {(1 - 0.25):.2f}")
print(f"        → 难题 (prompt 1) 获得 {(1-0.25)/(1-0.75):.1f}x 更大权重")

### 公式 8：损失归约（Reduction）

将 token-level 的损失聚合为一个标量：

$$\mathcal{L} = \sum_b \frac{\sum_t (\mathcal{L}_{b,t} \cdot \text{mask}_{b,t})}{1000}$$

- 先对每个样本内的所有 token 损失求和（通过 `response_mask` 过滤 padding 位置）
- 除以 1000 做缩放（避免长序列的损失值过大）
- 最后对 batch 内所有样本求和

In [ ]:
# 先对每个样本在序列维度 (token-level) 求和, 再除以 1000 做缩放
pg_loss = torch.sum(pg_losses * response_mask, dim=1) / 1000.0    # (bs,) 每个样本的总损失

# 最后对所有样本求和, 得到 batch 级的标量损失
pg_loss = torch.sum(pg_loss)                                       # scalar

print(f"[公式8] pg_loss (归约后): {pg_loss.item():.6f}")

### 公式 9：最终策略损失

最终的策略损失由三部分组成：

$$\mathcal{L}_{\text{policy}} = \mathcal{L}_{\text{pg}} - \beta_{\text{ent}} \cdot H(\pi) + \beta_{\text{kl}} \cdot D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$$

- $\mathcal{L}_{\text{pg}}$：前面的 NFT 策略梯度损失
- $H(\pi)$：**策略熵**，衡量策略的随机性。减去熵项等于鼓励策略保持一定的探索性，避免过早收敛
- $D_{\text{KL}}$：**KL 散度正则**，防止当前策略偏离参考策略太远（本 demo 中 $\beta_{\text{kl}}=0$，跳过）

In [ ]:
# 计算策略熵 H(π): 衡量策略的随机性/探索程度
entropy = actor.get_entropy(responses)                                        # (bs, seq)
entropy_loss = (entropy * response_mask).sum() / response_mask.sum()          # 加权均值

# 最终策略损失 = 策略梯度损失 - 熵正则项 (鼓励探索)
policy_loss = pg_loss - entropy_loss * ENTROPY_COEFF

# 注: KL 散度项在本 demo 中设为 0, 实际训练中可选启用
# 如果 KL_COEFF > 0, 会加入 policy_loss += kl_loss * KL_COEFF
# 用于防止当前策略偏离参考策略太远

print(f"[公式9] L_policy = L_pg - β·H(π)")
print(f"        pg_loss:     {pg_loss.item():.6f}")
print(f"        entropy:     {entropy_loss.item():.4f}")
print(f"        policy_loss: {policy_loss.item():.6f}")

### 公式 10：反向传播与参数更新

最后一步是标准的梯度下降流程：
1. **清零梯度** — 清除上一轮的累积梯度
2. **反向传播** — 通过链式法则计算所有参数梯度
3. **梯度裁剪** — 限制梯度范数不超过 1.0，防止训练不稳定
4. **参数更新** — Adam 优化器执行一步参数更新

In [ ]:
# 清零梯度
optimizer.zero_grad()

# 反向传播: 计算所有参数的梯度
policy_loss.backward()

# 梯度裁剪: 防止梯度爆炸 (对应 dp_actor.py:182-186)
grad_norm = torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=1.0)

# 参数更新
optimizer.step()

print(f"[更新] grad_norm: {grad_norm.item():.4f}")

## 验证：更新后 ratio 变化方向

训练完成后，我们检查模型参数的更新效果是否符合 NFT 的预期：
- **正样本**的 ratio 应该 > 1.0（模型更倾向于生成正确回答）
- **负样本**的 ratio 应该 < 1.0（模型抑制了错误回答的生成概率）

In [ ]:
# 用更新后的模型重新计算 log_probs 和 ratio
with torch.no_grad():
    new_log_probs = actor.get_log_probs(responses)                          # 更新后的 log_probs
    new_ratio = torch.exp(new_log_probs - old_log_probs)                    # 更新后的 ratio

# 观察 ratio 的整体偏移量
ratio_shift = (new_ratio - 1.0).mean()
print(f"[验证] 更新后 ratio 均值偏移: {ratio_shift.item():+.6f}")

# 分别检查正样本和负样本的 ratio 变化方向
correct_ratio_mean = new_ratio[correct_mask.squeeze().bool()].mean().item()   # 正样本 ratio 均值
wrong_ratio_mean   = new_ratio[~correct_mask.squeeze().bool()].mean().item()  # 负样本 ratio 均值

print(f"[验证] 正样本 ratio 均值: {correct_ratio_mean:.4f} (期望 > 1.0, 被鼓励)")
print(f"[验证] 负样本 ratio 均值: {wrong_ratio_mean:.4f} (期望 < 1.0, 被抑制)")

# 如果正样本 ratio > 负样本 ratio，说明 NFT 成功地同时利用了正负样本
if correct_ratio_mean > wrong_ratio_mean:
    print(f"[验证] ✅ NFT 成功同时利用了正样本和负样本!")
else:
    print(f"[验证] ❌ 未观察到预期效果")

## 对比实验：NFT vs RFT

最后我们做一个对比实验，在同一批数据上分别用 **NFT**（`neg_weight=1.0`）和 **RFT**（`neg_weight=0.0`）训练，观察两种方法对正样本和负样本 ratio 的不同影响：

- **NFT**：同时推高正样本 ratio、推低负样本 ratio
- **RFT**：只推高正样本 ratio，负样本 ratio 几乎不变（因为直接丢弃了负样本）

In [ ]:
print("\n" + "=" * 70)
print("对比实验: NFT vs RFT (同一批数据, 不同 neg_weight)")
print("=" * 70)

for neg_w, name in [(1.0, "NFT"), (0.0, "RFT")]:
    # 从旧策略的初始状态重新创建模型
    model = ToyPolicy(VOCAB_SIZE)
    model.load_state_dict(actor_old.state_dict())
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # 计算新的 log_probs 和 ratio
    lp = model.get_log_probs(responses)
    r = torch.exp((lp - old_log_probs).clamp(max=10.0))
    cr = torch.clamp(correct_rate, min=0.02, max=0.98)
    nr = 1.0 - cr

    if neg_w < 0:
        # DAPO 分支: 标准 PPO clipped surrogate (简化演示)
        adv = advantages
        loss_per_token = -torch.min(r * adv, torch.clamp(r, 0.8, 1.2) * adv)
        loss = (loss_per_token * response_mask).sum() / response_mask.sum()
    else:
        # NFT / RFT 分支: 通过隐式负策略计算损失
        proxy = (1.0 - cr * r) / (1.0 - cr)                              # 隐式负策略
        neg_l = -torch.log(clamp_preserve_grad(proxy, min_val=1.0))      # 负样本损失
        pos_l = -torch.log(clamp_preserve_grad(r, min_val=0.0))          # 正样本损失
        combined = neg_l * (1.0 - correct_mask) * neg_w + pos_l * correct_mask  # 组合损失
        if NORMALIZE == 1:
            combined = combined * nr                                     # 难度感知归一化
        loss = torch.sum(combined * response_mask) / 1000.0              # 归约

    # 反向传播并更新参数
    opt.zero_grad()
    loss.backward()
    opt.step()

    # 验证更新后的 ratio 变化
    with torch.no_grad():
        new_r = torch.exp(model.get_log_probs(responses) - old_log_probs)
        cr_mean = new_r[correct_mask.squeeze().bool()].mean().item()     # 正样本 ratio 均值
        wr_mean = new_r[~correct_mask.squeeze().bool()].mean().item()    # 负样本 ratio 均值

    print(f"  {name:4s} (neg_weight={neg_w}): "
          f"正样本 ratio={cr_mean:.4f}, 负样本 ratio={wr_mean:.4f}, "
          f"差值={cr_mean - wr_mean:+.4f}")

print(f"\n结论: NFT 通过隐式负策略同时推高正样本 & 推低负样本,")
print(f"      RFT 只利用正样本, 负样本的 ratio 几乎不变。")